# 02 – Removing redundant peptides with CD-HIT (90% identity)

**Why?** DBAASP contains many near-identical peptides: series of single-residue variants, truncations, repeated entries. If we leave them in, one peptide family that was studied 100 times counts 100 times, and group comparisons (and especially amino-acid enrichment) mostly describe those big families.

**Method.** [CD-HIT](https://sites.google.com/view/cd-hit) sorts the sequences by length and walks through them from the longest. A sequence that is ≥ 90% identical to an earlier one joins that cluster; otherwise it starts a new cluster. The first (longest) sequence of each cluster is the **representative**; we keep only representatives for the analysis.

Settings: `-c 0.9` (90% identity) and `-n 5` (word size 5, the value CD-HIT recommends for thresholds of 0.7–1.0).

Two practical details for peptides:
* CD-HIT throws away sequences whose length is `<= -l`, so we pass `-l 4` to keep peptides of 5 residues and longer.
* Word size 5 cannot handle peptides shorter than 5 residues. For peptides of 2–4 residues we simply merge **identical** sequences. This is consistent with 90% identity: for peptides of up to 9 residues, 90% identity already means "identical".

Input: `results/clean_peptides.csv` (notebook 01). Output: `results/clean_peptides_clustered.csv` (all rows, with cluster columns) and the CD-HIT files in `results/cdhit/`.

In [1]:
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IDENTITY, WORD_SIZE, MIN_LEN = 0.90, 5, 5
CD = Path("../results/cdhit")
CD.mkdir(exist_ok=True)

df = pd.read_csv("../results/clean_peptides.csv")
print(df.shape)

(18196, 24)


## 1. Split by length and write the FASTA file

In [2]:
long_pep = df[df["length"] >= MIN_LEN]
short_pep = df[df["length"] < MIN_LEN]
print(f"{len(long_pep)} peptides with >= {MIN_LEN} residues go to CD-HIT")
print(f"{len(short_pep)} peptides with < {MIN_LEN} residues are merged by identical sequence")

with open(CD / "input.fasta", "w") as f:
    for r in long_pep.itertuples():
        f.write(f">{r.ID}\n{r.sequence}\n")

17577 peptides with >= 5 residues go to CD-HIT
619 peptides with < 5 residues are merged by identical sequence


## 2. Run CD-HIT

In [3]:
if shutil.which("cd-hit") is None:
    raise SystemExit("cd-hit not found. Install it with `sudo apt install cd-hit` "
                     "or `conda install -c bioconda cd-hit`, then re-run this notebook.")

cmd = ["cd-hit", "-i", str(CD / "input.fasta"), "-o", str(CD / "representatives.fasta"),
       "-c", str(IDENTITY), "-n", str(WORD_SIZE), "-l", str(MIN_LEN - 1),
       "-d", "0", "-M", "0", "-T", "1"]     # -T 1 = single thread (deterministic)
print(" ".join(cmd))
run = subprocess.run(cmd, capture_output=True, text=True, check=True)
(CD / "cdhit_run.log").write_text(run.stdout)
print(run.stdout.strip().splitlines()[-6:])

cd-hit -i ../results/cdhit/input.fasta -o ../results/cdhit/representatives.fasta -c 0.9 -n 5 -l 4 -d 0 -M 0 -T 1


['Approximated maximum memory consumption: 80M', 'writing new database', 'writing clustering information', 'program completed !', '', 'Total CPU time 0.24']


## 3. Read the clusters
Each line of the `.clstr` file is a cluster member; the member marked `*` is the representative, the others show their % identity to it.

In [4]:
members, cid = [], None
for line in open(CD / "representatives.fasta.clstr"):
    if line.startswith(">Cluster"):
        cid = f"C{int(line.split()[1])}"
        continue
    m = re.match(r"\d+\s+(\d+)aa, >(\S+?)\.\.\. (\*|at .*?(\d+\.\d+)%)", line)
    members.append({"ID": int(m.group(2)), "cluster_id": cid, "is_representative": m.group(3) == "*",
                    "identity": 100.0 if m.group(3) == "*" else float(m.group(4))})
cl = pd.DataFrame(members)

# sanity checks on the CD-HIT output
assert set(cl["ID"]) == set(long_pep["ID"]), "CD-HIT dropped or invented sequences"
assert cl["ID"].is_unique
assert (cl.groupby("cluster_id")["is_representative"].sum() == 1).all(), "each cluster needs exactly one representative"
assert cl["identity"].min() >= 90, "a member is below the 90% identity threshold"
print(f"{len(cl)} peptides in {cl['cluster_id'].nunique()} clusters; lowest identity to a representative: {cl['identity'].min():.1f}%")

17577 peptides in 9310 clusters; lowest identity to a representative: 90.2%


## 4. Merge identical short peptides and combine

In [5]:
# short peptides: one cluster per identical sequence; representative = lowest ID
sh = short_pep[["ID", "sequence"]].sort_values("ID").copy()
sh["cluster_id"] = "S" + sh.groupby("sequence").ngroup().astype(str)
sh["is_representative"] = ~sh["sequence"].duplicated()
sh["identity"] = 100.0

clusters = pd.concat([cl, sh[["ID", "cluster_id", "is_representative", "identity"]]], ignore_index=True)
out = df.merge(clusters, on="ID", how="left", validate="one_to_one")

assert out["cluster_id"].notna().all()
assert (out.groupby("cluster_id")["is_representative"].sum() == 1).all()
out["cluster_size"] = out.groupby("cluster_id")["ID"].transform("size")
out["cluster_n_origins"] = out.groupby("cluster_id")["origin"].transform("nunique")
print(f"{len(out)} peptides -> {out['cluster_id'].nunique()} clusters -> {out['is_representative'].sum()} representatives")

18196 peptides -> 9622 clusters -> 9622 representatives


## 5. What did redundancy removal do?

In [6]:
nr = out[out["is_representative"]]
tab = pd.DataFrame({"before": out["origin"].value_counts(), "after (representatives)": nr["origin"].value_counts()})
tab.loc["Total"] = tab.sum()
tab["kept_%"] = (100 * tab["after (representatives)"] / tab["before"]).round(1)
tab

,before,after (representatives),kept_%
origin,,,
Synthetic,14957,7314,48.9
Natural,3239,2308,71.3
Total,18196,9622,52.9


In [7]:
sizes = out.drop_duplicates("cluster_id")["cluster_size"]
print("cluster size: median", int(sizes.median()), "| clusters with 1 peptide:", int((sizes == 1).sum()),
      "| largest cluster:", int(sizes.max()))

mixed = out.drop_duplicates("cluster_id")["cluster_n_origins"].eq(2)
print(f"clusters containing BOTH natural and synthetic peptides: {mixed.sum()} of {len(sizes)} ({100*mixed.mean():.1f}%)")

# CD-HIT compares a shorter sequence with the part of the longer one it aligns to, so fragments
# of a longer peptide join its cluster. How common is that?
rep_len = out[out["is_representative"]].set_index("cluster_id")["length"]
non_rep = out[~out["is_representative"]]
frag = (non_rep["length"] < 0.9 * non_rep["cluster_id"].map(rep_len)).mean()
print(f"removed peptides that were shorter than 90% of their representative: {100*frag:.0f}%")
print("\nmedian length before -> after:")
print(pd.DataFrame({"before": out.groupby("origin")["length"].median(),
                    "after": nr.groupby("origin")["length"].median()}))

cluster size: median 1 | clusters with 1 peptide: 7136 | largest cluster: 108
clusters containing BOTH natural and synthetic peptides: 500 of 9622 (5.2%)
removed peptides that were shorter than 90% of their representative: 47%

median length before -> after:
           before  after
origin                  
Natural      25.0   25.0
Synthetic    15.0   15.0


In [8]:
COLORS = {"Natural": "#2a9d8f", "Synthetic": "#e76f51"}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
x = np.arange(2); w = 0.38
for k, (label, data) in enumerate([("before", out), ("after (non-redundant)", nr)]):
    counts = data["origin"].value_counts().reindex(["Natural", "Synthetic"])
    axes[0].bar(x + (k - 0.5) * w, counts.values, w, label=label, color=["#2a9d8f", "#e76f51"], alpha=1 if k else 0.4)
axes[0].set_xticks(x); axes[0].set_xticklabels(["Natural", "Synthetic"]); axes[0].set_ylabel("peptides")
axes[0].set_title("Peptides per origin", fontsize=10)
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color="grey", alpha=0.4), plt.Rectangle((0, 0), 1, 1, color="grey")],
               labels=["before", "after"], frameon=False)
bins = np.arange(0, 101, 2)
for o in ["Natural", "Synthetic"]:
    axes[1].hist(out.loc[out.origin == o, "length"].clip(upper=100), bins=bins, histtype="step", ls="--", color=COLORS[o])
    axes[1].hist(nr.loc[nr.origin == o, "length"].clip(upper=100), bins=bins, histtype="step", color=COLORS[o], label=o)
axes[1].set_xlabel("length (residues, capped at 100)"); axes[1].set_title("Length: before (dashed) vs after (solid)", fontsize=10)
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../figures/redundancy_before_after.png", dpi=150)
plt.show()

## Save

In [9]:
cols = list(df.columns) + ["cluster_id", "is_representative", "cluster_size", "cluster_n_origins"]
out[cols].to_csv("../results/clean_peptides_clustered.csv", index=False)

summary = tab.reset_index().rename(columns={"index": "origin"})
summary.to_csv("../results/redundancy_summary.csv", index=False)
print("saved", out[cols].shape)

saved (18196, 28)


## Notes for the methods / limitations section
* Redundancy was removed with CD-HIT 4.8.1 at 90% identity (`-c 0.9 -n 5 -l 4`), clustering natural and synthetic peptides **together**. Natural and synthetic peptides rarely fall in the same cluster (see step 5), so this hardly moves peptides between groups.
* CD-HIT keeps the **longest** sequence of each cluster and compares a shorter sequence with the part of the longer one it matches, so fragments of a longer peptide are merged into it. The non-redundant set is therefore slightly biased towards longer peptides.
* Peptides of 2–4 residues cannot be clustered with word size 5, so only identical sequences were merged for them.
* All later notebooks use only the representatives (`is_representative == True`).